# Chapter 5 — Exploratory Data Analysis: Cleaning, Profiling & Feature Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part1_foundations/ch05_exploratory_data_analysis/ch05_notebook.ipynb)

Companion notebook for **Chapter 5** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- EDA has four steps: collect, clean, profile and visualize. Never skip it.
- Know each column's type. It decides the statistics, the charts and the encoding you can use.
- Describe numbers by center (mean, median, mode), spread (variance, std, IQR) and shape (skewness).
- Fill missing values in a way that suits the data: mean or median, forward fill, interpolation, or KNN.
- Find outliers with the Z-score rule (|z| > 3) or the IQR rule (1.5 × IQR), then remove, cap or transform them.
- Good features are often engineered: bins, interactions, log transforms, date parts and rare-category grouping.
- A feature that predicts the target almost perfectly may be data leakage. Check it is available at prediction time.

## Example 1: Profiling a Dataset

This is the first thing you do with any new dataset. The program builds a small passenger table, splits the columns into numbers and categories, then measures the center and spread of 'age'. Look at the skewness lines. Age is roughly symmetric, so its mean and median are close. Fare is strongly right-skewed: its mean (31) is well above its median (21) because a few people paid very high fares.

In [1]:
# Chapter 5 - Example 1: Profiling a dataset (data types, center, spread, shape)
import numpy as np
import pandas as pd
from scipy.stats import skew

# A small passenger dataset, built in code so it runs anywhere (no download)
rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    "age":      np.round(rng.normal(30, 12, n).clip(1, 75)),
    "fare":     np.round(rng.lognormal(3.0, 0.9, n), 2),   # long right tail
    "pclass":   rng.choice([1, 2, 3], n, p=[0.25, 0.25, 0.5]),
    "sex":      rng.choice(["male", "female"], n, p=[0.6, 0.4]),
    "embarked": rng.choice(["S", "C", "Q"], n, p=[0.7, 0.2, 0.1]),
})

# 1. What kind of data is each column?
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]
print("Numerical columns  :", num_cols)
print("Categorical columns:", cat_cols)

# 2. Central tendency and dispersion for 'age'
age = df["age"]
print("\n--- Age: center and spread ---")
print(f"Mean   : {age.mean():.2f}")
print(f"Median : {age.median():.2f}")
print(f"Mode   : {age.mode()[0]:.0f}")
print(f"Var    : {age.var():.2f}")
print(f"Std    : {age.std():.2f}")
print(f"Range  : {age.max() - age.min():.0f}")

# 3. Five-point summary (min, Q1, median, Q3, max)
q = age.quantile([0, 0.25, 0.5, 0.75, 1.0])
print("\n--- Five-point summary of age ---")
for name, v in zip(["Min", "Q1", "Median", "Q3", "Max"], q):
    print(f"{name:7s}: {v:.1f}")

# 4. Skewness tells us which way the tail points
for col in ["age", "fare"]:
    s = skew(df[col])
    shape = "right-skewed" if s > 0.5 else "left-skewed" if s < -0.5 else "roughly symmetric"
    print(f"\nSkewness of {col}: {s:.2f} -> {shape}")
    print(f"  mean = {df[col].mean():.2f}, median = {df[col].median():.2f}")

# 5. Categorical columns: counts are the summary
print("\nPassengers per class:")
print(df["pclass"].value_counts().sort_index().to_string())

Numerical columns  : ['age', 'fare', 'pclass']
Categorical columns: ['sex', 'embarked']

--- Age: center and spread ---
Mean   : 29.67
Median : 29.00
Mode   : 25
Var    : 112.60
Std    : 10.61
Range  : 61

--- Five-point summary of age ---
Min    : 4.0
Q1     : 22.0
Median : 29.0
Q3     : 36.2
Max    : 65.0

Skewness of age: 0.27 -> roughly symmetric
  mean = 29.67, median = 29.00

Skewness of fare: 3.29 -> right-skewed
  mean = 31.03, median = 21.21

Passengers per class:
pclass
1     50
2     45
3    105


## Example 2: Cleaning — Missing Values, Duplicates and Outliers

We plant three kinds of problem in a clean dataset and then fix them. Compare the five ways of filling the four missing weights. Mean and median give every row the same value. KNN imputation uses each person's height, so a shorter person gets a lower weight, which is the most sensible result. Both the Z-score and IQR tests catch the 250 kg typing error. The IQR test also flags one genuinely light person (43.4 kg), a reminder that not every outlier is a mistake. The last lines show how removing, capping or log-transforming the outliers changes the numbers.

In [2]:
# Chapter 5 - Example 2: Cleaning data - missing values, duplicates and outliers
import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.impute import KNNImputer

rng = np.random.default_rng(7)
n = 100
df = pd.DataFrame({"height": rng.normal(170, 8, n).round(1)})
# taller people tend to be heavier, plus some random variation
df["weight"] = (0.9 * (df["height"] - 170) + 70 + rng.normal(0, 4, n)).round(1)

# Plant some problems on purpose
df.loc[[3, 15, 40, 77], "weight"] = np.nan   # missing values
df.loc[60, "weight"] = 250.0                 # a typing error -> outlier
df = pd.concat([df, df.iloc[[5, 9]]], ignore_index=True)  # 2 duplicate rows

print("Rows:", len(df), "| missing weights:", df["weight"].isna().sum(),
      "| duplicates:", df.duplicated().sum())

# 1. Remove duplicates (keep the first copy)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print("After drop_duplicates:", len(df), "rows")

# 2. Compare ways of filling the gaps
true_missing = df["weight"].isna()
filled = {
    "Mean":          df["weight"].fillna(df["weight"].mean()),
    "Median":        df["weight"].fillna(df["weight"].median()),
    "Forward fill":  df["weight"].ffill(),
    "Interpolation": df["weight"].interpolate(),
    "KNN (k=3)":     pd.Series(KNNImputer(n_neighbors=3).fit_transform(df)[:, 1]),
}
print("\nFilled values for the 4 missing rows:")
for name, s in filled.items():
    print(f"  {name:14s}", np.round(s[true_missing].values, 1))
print("  Heights of those rows:", df.loc[true_missing, "height"].values)

df["weight"] = filled["KNN (k=3)"]   # KNN uses height, so it is the most sensible

# 3. Outliers by Z-score (|z| > 3)
z = zscore(df["weight"])
print("\nZ-score outliers:", df.loc[np.abs(z) > 3, "weight"].tolist())

# 4. Outliers by IQR (beyond 1.5 x IQR from the quartiles)
q1, q3 = df["weight"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR fences: [{lo:.1f}, {hi:.1f}]")
print("IQR outliers:", df.loc[(df["weight"] < lo) | (df["weight"] > hi), "weight"].tolist())

# 5. Three ways to deal with the outlier
removed = df[(df["weight"] >= lo) & (df["weight"] <= hi)]
capped = df["weight"].clip(df["weight"].quantile(0.05), df["weight"].quantile(0.95))
logged = np.log1p(df["weight"])
print(f"\nMean weight  raw: {df['weight'].mean():.2f}  removed: {removed['weight'].mean():.2f}"
      f"  winsorized: {capped.mean():.2f}")
print(f"Max after log transform: {logged.max():.2f} (raw max {df['weight'].max():.0f})")

Rows: 102 | missing weights: 4 | duplicates: 2
After drop_duplicates: 100 rows

Filled values for the 4 missing rows:
  Mean           [70.2 70.2 70.2 70.2]
  Median         [68.4 68.4 68.4 68.4]
  Forward fill   [68.7 70.  65.3 73. ]
  Interpolation  [66.  65.6 66.8 71.2]
  KNN (k=3)      [63.6 74.2 73.5 70.1]
  Heights of those rows: [162.9 175.6 170.9 167.1]

Z-score outliers: [250.0]
IQR fences: [49.4, 88.1]
IQR outliers: [43.4, 250.0]

Mean weight  raw: 70.20  removed: 68.64  winsorized: 68.61
Max after log transform: 5.53 (raw max 250)


## Example 3: Advanced EDA — Target Analysis, Encoding, Features and Leakage

This mirrors the classic Titanic case study, using made-up data. Grouping by class and sex shows who was more likely to survive. We then encode categories, engineer family size, age groups and a log fare, and rank every feature by its correlation with survival. One column, 'lifeboat_assigned', has a correlation of 0.97. It looks like a perfect predictor, but it is only known after the event. The program flags it as data leakage.

In [3]:
# Chapter 5 - Example 3: Advanced EDA - correlation, target-wise analysis,
# encoding, feature engineering and a data-leakage check
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
n = 600
pclass = rng.choice([1, 2, 3], n, p=[0.25, 0.25, 0.5])
sex = rng.choice(["male", "female"], n, p=[0.6, 0.4])
age = rng.normal(30, 12, n).clip(1, 75).round()
sibsp = rng.poisson(0.5, n)
parch = rng.poisson(0.4, n)
fare = np.round(np.where(pclass == 1, 80, np.where(pclass == 2, 25, 10))
                * rng.lognormal(0, 0.4, n), 2)
# survival chance depends on sex, class and age (like the real Titanic)
logit = 1.2 * (sex == "female") - 0.9 * (pclass - 2) - 0.02 * (age - 30) - 0.4
survived = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
df = pd.DataFrame(dict(survived=survived, pclass=pclass, sex=sex, age=age,
                       sibsp=sibsp, parch=parch, fare=fare))
# A leaky column: recorded AFTER the voyage, so it secretly contains the answer
df["lifeboat_assigned"] = np.where(survived == 1, 1, (rng.random(n) < 0.03).astype(int))

# 1. Target-wise EDA: survival rate by group
print("Survival rate by class:")
print(df.groupby("pclass")["survived"].mean().round(2).to_string())
print("\nSurvival rate by sex:")
print(df.groupby("sex")["survived"].mean().round(2).to_string())
print("\nMedian fare by outcome:")
print(df.groupby("survived")["fare"].median().to_string())

# 2. Encoding categorical data
df["sex_label"] = df["sex"].map({"female": 0, "male": 1})          # label encoding
onehot = pd.get_dummies(df["pclass"], prefix="class", dtype=int)  # one-hot
print("\nOne-hot columns:", onehot.columns.tolist())

# 3. Feature engineering
df["family_size"] = df["sibsp"] + df["parch"] + 1                  # interaction
df["age_group"] = pd.cut(df["age"], bins=[0, 12, 18, 60, 100],
                         labels=["child", "teen", "adult", "senior"])  # binning
df["log_fare"] = np.log1p(df["fare"])                              # transformation
print("\nSurvival by age group:")
print(df.groupby("age_group", observed=True)["survived"].mean().round(2).to_string())

# 4. Correlation with the target - and a leakage warning
num = df[["survived", "pclass", "sex_label", "age", "family_size",
          "log_fare", "lifeboat_assigned"]]
corr = num.corr()["survived"].drop("survived").round(2).sort_values(key=abs, ascending=False)
print("\nCorrelation with 'survived':")
print(corr.to_string())
for col, r in corr.items():
    if abs(r) > 0.8:
        print(f"\nWARNING: '{col}' has r = {r} with the target. Is it known at prediction"
              " time? If not, it is DATA LEAKAGE - drop it.")

Survival rate by class:
pclass
1    0.73
2    0.51
3    0.31

Survival rate by sex:
sex
female    0.60
male      0.36

Median fare by outcome:
survived
0    14.10
1    26.56

One-hot columns: ['class_1', 'class_2', 'class_3']

Survival by age group:
age_group
child     0.69
teen      0.52
adult     0.44
senior    0.40

Correlation with 'survived':
lifeboat_assigned    0.97
pclass              -0.33
log_fare             0.27
sex_label           -0.24
age                 -0.15
family_size          0.00

